# Evaluation P@K, R@K, F1@K
Notebook so sánh model hybrid với các model riêng lẻ theo logic production.

In [1]:
# %pip install pandas numpy sqlalchemy pymysql python-dotenv

In [2]:
import os
from collections import defaultdict
from urllib.parse import parse_qsl, quote_plus, urlencode, urlparse
import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text

In [3]:
def build_sqlalchemy_url(db_url: str, username: str, password: str) -> str:
    if db_url.startswith('jdbc:'):
        db_url = db_url[5:]
    p = urlparse(db_url)
    q = dict(parse_qsl(p.query, keep_blank_values=True))
    mapped = {}
    if 'useSSL' in q:
        mapped['ssl_disabled'] = str((q['useSSL'].lower() != 'true')).lower()
    mapped.setdefault('charset', 'utf8mb4')
    mapped.setdefault('connect_timeout', '10')
    return f"mysql+pymysql://{quote_plus(username)}:{quote_plus(password)}@{p.hostname}:{p.port or 3306}/{p.path.lstrip('/')}?{urlencode(mapped)}"

load_dotenv(find_dotenv(usecwd=True), override=False)
engine = create_engine(build_sqlalchemy_url(os.getenv('DB_URL'), os.getenv('DB_USERNAME'), os.getenv('DB_PASSWORD')), pool_pre_ping=True)
print('connected')

connected


In [4]:
premium_song_type = int(os.getenv('PREMIUM_SONG_TYPE', '1'))
premium_boost_multiplier = float(os.getenv('PREMIUM_BOOST_MULTIPLIER', '1.35'))
premium_min_slots = int(os.getenv('PREMIUM_MIN_SLOTS', '2'))
ACTION_WEIGHT = {0: 1.0, 1: 3.0, 2: 4.0, 3: 2.0, 4: 5.0}

In [5]:
inter = pd.read_sql(text('SELECT user_id, song_id, action_type, created_at FROM interaction_song WHERE user_id IS NOT NULL'), engine)
inter_artist = pd.read_sql(text('SELECT user_id, artist_id, COALESCE(interaction_count,0) AS interaction_count FROM interaction_artist WHERE user_id IS NOT NULL'), engine)
song = pd.read_sql(text('SELECT id AS song_id, type, deleted_at FROM song'), engine)
artist_song = pd.read_sql(text('SELECT song_id, artist_id FROM artist_song'), engine)
genre_song = pd.read_sql(text('SELECT song_id, genre_id FROM genre_song'), engine)
audio_q = pd.read_sql(text('SELECT song_id, SUM(CASE WHEN type = 1 THEN 1 ELSE 0 END) AS high_q_cnt, SUM(CASE WHEN url IS NOT NULL AND url <> "" THEN 1 ELSE 0 END) AS playable_cnt FROM audio_quality GROUP BY song_id'), engine)

inter['created_at'] = pd.to_datetime(inter['created_at'], errors='coerce')
inter = inter.dropna(subset=['created_at', 'user_id', 'song_id'])
active_songs = set(song[song['deleted_at'].isna()]['song_id'])
audio_map = audio_q.set_index('song_id')[['high_q_cnt', 'playable_cnt']].to_dict('index')
premium_set = set(song[song['type'] == premium_song_type]['song_id'])

In [6]:
cut = inter['created_at'].quantile(0.8)
train = inter[inter['created_at'] <= cut].copy()
test = inter[inter['created_at'] > cut].copy()
valid_users = set(train['user_id'].unique()) & set(test['user_id'].unique())
train = train[train['user_id'].isin(valid_users)]
test = test[test['user_id'].isin(valid_users)]
print('train', len(train), 'test', len(test), 'users', len(valid_users))

train 11784 test 7680 users 301


In [7]:
train['w'] = train['action_type'].map(ACTION_WEIGHT).fillna(1.0)
train_seen = train.groupby('user_id')['song_id'].apply(set).to_dict()
truth = test.groupby('user_id')['song_id'].apply(set).to_dict()
pop = train.groupby('song_id')['w'].sum().to_dict()
user_hist = train.groupby('user_id')['song_id'].apply(list).to_dict()
song_users = defaultdict(set)
for row in train[['user_id', 'song_id']].itertuples(index=False):
    song_users[row.song_id].add(row.user_id)
song_genres = genre_song.groupby('song_id')['genre_id'].apply(set).to_dict()
song_artists = artist_song.groupby('song_id')['artist_id'].apply(set).to_dict()
artist_pref_map = defaultdict(dict)
for row in inter_artist.itertuples(index=False):
    artist_pref_map[row.user_id][row.artist_id] = float(row.interaction_count)

In [8]:
def apply_audio_filter_and_boost(scores):
    out = {}
    for sid, sc in scores.items():
        if sid not in active_songs:
            continue
        aq = audio_map.get(sid, {'high_q_cnt': 0, 'playable_cnt': 0})
        if float(aq['playable_cnt']) <= 0:
            continue
        q_boost = 1.0 + min(float(aq['playable_cnt']), 3.0) * 0.05 + min(float(aq['high_q_cnt']), 2.0) * 0.08
        out[sid] = float(sc) * q_boost
    return out

def apply_premium_boost(scores):
    return {sid: (float(sc) * premium_boost_multiplier if sid in premium_set else float(sc)) for sid, sc in scores.items()}

def topk_with_premium_slots(scores, k):
    ranked = [sid for sid, _ in sorted(scores.items(), key=lambda t: t[1], reverse=True)]
    top = ranked[:k]
    premium_top = [sid for sid in top if sid in premium_set]
    need = max(0, premium_min_slots - len(premium_top))
    if need > 0:
        backup = [sid for sid in ranked[k:] if sid in premium_set][:need]
        non = [sid for sid in top if sid not in premium_set]
        top = (premium_top + backup + non)[:k]
    return top

In [9]:
all_song_ids = set(song['song_id'].unique())

def scorer_pop(u, exclude):
    s = {k: v for k, v in pop.items() if k not in exclude}
    s = apply_audio_filter_and_boost(s)
    return apply_premium_boost(s)

def scorer_content(u, exclude):
    hist = user_hist.get(u, [])[:50]
    pg, pa = set(), set()
    for sid in hist:
        pg |= song_genres.get(sid, set())
        pa |= song_artists.get(sid, set())
    s = {}
    for sid in all_song_ids:
        if sid in exclude:
            continue
        go = len(pg & song_genres.get(sid, set()))
        ao = len(pa & song_artists.get(sid, set()))
        if go or ao:
            s[sid] = ao * 2 + go
    s = apply_audio_filter_and_boost(s)
    return apply_premium_boost(s)

def scorer_collab(u, exclude):
    hist = user_hist.get(u, [])[:30]
    s = defaultdict(float)
    for sid in hist:
        for ou in song_users.get(sid, set()):
            if ou == u:
                continue
            for csid in train_seen.get(ou, set()):
                if csid != sid and csid not in exclude:
                    s[csid] += 1.0
    s = apply_audio_filter_and_boost(s)
    return apply_premium_boost(s)

def scorer_artist(u, exclude):
    ap = artist_pref_map.get(u, {})
    s = defaultdict(float)
    if not ap:
        return {}
    for sid, artists in song_artists.items():
        if sid in exclude:
            continue
        for a in artists:
            s[sid] += ap.get(a, 0.0)
    s = apply_audio_filter_and_boost(s)
    return apply_premium_boost(s)

def scorer_hybrid(u, exclude):
    c = scorer_collab(u, exclude)
    t = scorer_content(u, exclude)
    p = scorer_pop(u, exclude)
    a = scorer_artist(u, exclude)
    s = defaultdict(float)
    for k, v in c.items():
        s[k] += 0.45 * v
    for k, v in t.items():
        s[k] += 0.25 * v
    for k, v in p.items():
        s[k] += 0.10 * v
    for k, v in a.items():
        s[k] += 0.20 * v
    s = apply_audio_filter_and_boost(s)
    return apply_premium_boost(s)

In [10]:
def precision_recall_f1_at_k(reco, truth_set, k):
    rec_k = reco[:k]
    if not rec_k:
        return 0.0, 0.0, 0.0
    hit = len(set(rec_k) & set(truth_set))
    p = hit / k
    r = hit / len(truth_set) if truth_set else 0.0
    f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
    return p, r, f1

def evaluate(name, scorer, ks=(5, 10, 20), max_users=500):
    users = list(truth.keys())[:max_users]
    rows = []
    for k in ks:
        p_vals, r_vals, f_vals = [], [], []
        for u in users:
            ex = train_seen.get(u, set())
            gt = truth.get(u, set())
            scores = scorer(u, ex)
            rec = topk_with_premium_slots(scores, k)
            p, r, f = precision_recall_f1_at_k(rec, gt, k)
            p_vals.append(p)
            r_vals.append(r)
            f_vals.append(f)
        rows.append({
            'model': name,
            'k': k,
            'P@K': float(np.mean(p_vals)),
            'R@K': float(np.mean(r_vals)),
            'F1@K': float(np.mean(f_vals)),
            'users': len(users)
        })
    return pd.DataFrame(rows)

In [11]:
results = pd.concat([
    evaluate('popularity_only', scorer_pop),
    evaluate('content_only', scorer_content),
    evaluate('collab_only', scorer_collab),
    evaluate('artist_only', scorer_artist),
    evaluate('hybrid_prod_aligned', scorer_hybrid),
], ignore_index=True)
results.sort_values(['k', 'F1@K'], ascending=[True, False])

,model,k,P@K,R@K,F1@K,users
12,hybrid_prod_aligned,5,0.125581,0.116868,0.085058,301
0,popularity_only,5,0.124917,0.115869,0.083847,301
6,collab_only,5,0.112957,0.090073,0.070819,301
3,content_only,5,0.010631,0.003060,0.003600,301
9,artist_only,5,0.007973,0.002899,0.003314,301
13,hybrid_prod_aligned,10,0.106977,0.185091,0.093115,301
1,popularity_only,10,0.101329,0.188748,0.090591,301
7,collab_only,10,0.099336,0.171302,0.084235,301
4,content_only,10,0.009967,0.007346,0.005533,301
10,artist_only,10,0.007641,0.005296,0.004439,301
